# Homework 3: Vaccine Warehouse Binary

###  Bus 36109 "Advanced Decision Modeling with Python", Don Eisenstein
Don Eisenstein &copy; Copyright 2024, University of Chicago 

## Instructions

A vaccine is being produced at different manufacturing Plants to be distributed across the country to various Hospitals. Vaccines must travel through Warehouses along the way to a Hospital. 

Vaccines are transported in units of cases.  The capacity of a plant is in units of cases per week.  Each Warehouse must process and ship all the cases it receives each week.  Each Warehouse has a capacity of cases it can process each week and a processing cost to process each case.  Each Warehouse also has a fixed weekly operating cost that is incurred if it processes any cases of vaccine.  If a Warehouse processes no vaccine then its fixed cost is not incurred (that is, we can close a warehouse down and not incur its fixed cost). 

Each Hospital has a demand for vaccine cases each week.  The supply of vaccine to a hospital must not exceed its demand.  

The transportation cost is \$1 per vaccine case traveling one unit of distance between plants and warehouses and between warehouses and hospitals.

All facilites, distances and costs are described in an Airtable base.

Each manufacturing Plant must ship as many doses as possible in an attempt to meet, but not exceed, Hospital demand.  A manufacturing Plant can ship vaccines to multiple Warehouses, a Warehouse can ship vaccines to multiple Hospitals, and a Hospital can receive vaccines from multiple Warehouses.   

Your code should be robust, that is, it should NOT make any assumptions about the total plant and warehouse capacities, or total hospital demands. 

Your model should find the optimal vaccine flows from Plants to Warehouses and Warehouses to Hospitals to minimize the weekly transportation, processing and operating costs, while sending as much vaccine as is feasibly possible.  And in doing so, determining which Warehouses are Open and which Closed.

**IMPORTANT:** Model your flow of vaccine cases as a continuous variable.  That is, you can ship fractions of cases between facilities.

**NOTE** No single variable carrying flow should include the ENTIRE path from a Plant through a Warehouse and into a Hospital.   That is, you must have a set of simple flow variables from Plants to Warehouses, and another set of simple flow variables from Warehouses to Hospitals.

Follow the notebook to walk you through the solution in parts.  Insert your answer to each part into the notebook below the question for each part.   Turn in your completed notebook with all output visible.

# Your Solution

Insert your answer to each part into this notebook

In [12]:
import pulp
from pyairtable import Api
from pprint import pprint

In [2]:
AIRTABLE_API_TOKEN="Removed for Submission" 

BASE_ID="appWITdHvwKWupWZR" # The ID for the HW3 data table -- don't change this

api = Api(AIRTABLE_API_TOKEN)

<!-- BEGIN QUESTION -->

**1. In broad terms, what are the variables, objective and constraints of this problem? You don't need to list the entire formulation. Just describe the structure/characteristics of your model.**

In [3]:
# Variables are the arcs and the binary decision to have a particular warehouse open or closed: I'll call them Xp,w for the cases shipped from plant to warehouse and Yw,h for the cases shipped from warehouse to hospital, and Zw which will be either 0 (warehouse closed) or 1 ( warehouse open) 
# Objective is to first maximize demand served, and then given that, minmize total cost of that max delivery. So once we maximize demand, the objective to minimize cost formula would essentially be minimize cost of transportation, processing, and operating costs from plant to warehouse times X and that similar cost component from warehouse to hospital times Y plus the fixed costs of active warehouses.
# Constraints are demand served must be less than or equal to demand from the hospitals, the plant supply X must be less than or equal to plant Capacity, and the warehouse capacity must be greater than or equal to the amount shipped to the warehouse. Additionally the warehouse can only accept capcity if it is open. 

**2. Click on this link to access the data on AirTable: [AirTable Data](https://airtable.com/invite/l?inviteId=invCF2IEwn3KB68tR&inviteToken=8feebbc273e675e663a026e7a321221cedbdd4d893229610e2df40a2488647ef&utm_medium=email&utm_source=product_team&utm_content=transactional-alerts)** 

You may have to add access to this airtable base onto your token if you did not add permission to ALL bases to start.

<!-- END QUESTION -->

**3. Read in the `plants` table from AirTable. Store in an appropriate structure in Python.  Print out your Python structure. Verify that the data looks as expected.**

In [4]:
Plant_Table = "plants" # read in plants table
plants_table = api.table(BASE_ID, Plant_Table)
plants = plants_table.all()
#pprint(plants)

# make it dictionary format
plants_dict = {}
for p in plants:
    f = p['fields']
    name = f['name']
    cap = f['capacity']
    plants_dict[name] = float(cap)

print('num plants:', len(plants_dict))
print('plant example:', plants_dict[list(plants_dict.keys())[0]])


num plants: 6
plant example: 100.0


**4.  Now read in, print, and verify the hospitals table.** 

In [5]:
Hospital_Table = "hospitals" # read in hospitals table
hospitals_table = api.table(BASE_ID, Hospital_Table)
hospitals = hospitals_table.all()
#pprint(hospitals)

# make it dictionary format
hospitals_dict = {}
for h in hospitals:
    f = h['fields']
    name = f['name']
    demand = f['demand']
    hospitals_dict[name] = float(demand)

print('num hospitals:', len(hospitals_dict))
print('hospital example:', hospitals_dict[list(hospitals_dict.keys())[0]])


num hospitals: 10
hospital example: 40.0


**5.  Now read in, print, and verify the warehouse table.** 

In [6]:
Warehouse_Table = "warehouses" # read in warehouses table
warehouses_table = api.table(BASE_ID, Warehouse_Table)
warehouses = warehouses_table.all()
#pprint(warehouses_records)

# make it dictionary format
warehouses_dict = {}
for w in warehouses:
    f = w['fields']
    name = f['name']
    warehouses_dict[name] = {
        'capacity': float(f['capacity']),
        'processing_cost': float(f['processing_cost']),
        'fixed_cost': float(f['fixed_cost'])
    }

print('num warehouses:', len(warehouses_dict))
print('warehouse example:', warehouses_dict[list(warehouses_dict.keys())[0]])

num warehouses: 8
warehouse example: {'capacity': 100.0, 'processing_cost': 10.0, 'fixed_cost': 550.0}


**6.  Now read in, print, and verify the distances table.** 

In [7]:
Distance_Table = "distances" # read in distances table
distances_table = api.table(BASE_ID, Distance_Table)
distances = distances_table.all()
#pprint(distances)

# make it dictionary format
distances_dict = {}
for d in distances:
    f = d['fields']
    start = f['start']
    end = f['end']
    distances_dict[(start, end)] = float(f['distance'])
    
print('num distance pairings:', len(distances_dict))
first_key = list(distances_dict.keys())[0]
start, end = first_key
dist_val = distances_dict[first_key]
print("row:", {"start": start, "end": end, "distance": dist_val})


num distance pairings: 128
row: {'start': 'warehouse_5', 'end': 'hospital_3', 'distance': 284.0}


**7. Create a PuLP LpProblem object and store it in the variable `model`.** 

In [8]:
from pulp import *
model = LpProblem("Vaccine_Warehouse_Network",LpMaximize)

**8. Create and store your PuLP decision variables. Print out your Python structures that hold them.**


In [9]:
# binary variable z to represent if warehouse is open or closed
z = {}
for w in warehouses_dict:
    z[w] = pulp.LpVariable(f"z_{w}", cat = "Binary")
                           
# continuous variable x to represent transport arcs from plants to warehouses
x = {}
for p in plants_dict:
    for w in warehouses_dict:
        if (p,w) in distances_dict:
            x[(p,w)] = pulp.LpVariable(f"x_{p}_{w}", lowBound = 0, cat = "Continuous")

# continuous variable y to represent transport arcs from warehouses to hospitals
y = {}
for w in warehouses_dict:
    for h in hospitals_dict:
        if (w,h) in distances_dict:
            y[(w,h)] = pulp.LpVariable(f"y_{w}_{h}", lowBound = 0, cat = "Continuous")


**9. Add your objective function to your `model`.  Print your model to verify**

In [29]:
model = pulp.LpProblem('Vaccine_Netowrk_Max_Serve_Min_Cost', pulp.LpMinimize)

# define served amnount
served = pulp.lpSum(y[k] for k in y)

# define shipping cost for plant to warehouse and warehouse to hospital
shipping_cost = pulp.lpSum(distances_dict[(p,w)] * x[(p,w)] for (p,w) in x) + pulp.lpSum(distances_dict[(w,h)] * y[(w,h)] for (w,h) in y)
# define processing cost at the warehouse
processing_cost = 0
for w in warehouses_dict:
    inflow_w = pulp.lpSum(x[(p,w)] for p in plants_dict if (p,w) in x)
    #only account for this if there is actual inflow to the warehouse
    processing_cost += warehouses_dict[w]['processing_cost'] * inflow_w

# define fixed cost only if the plant is actually open
fixed_cost = pulp.lpSum(warehouses_dict[w]['fixed_cost'] * z[w] for w in warehouses_dict)

# total cost
total_cost = shipping_cost + processing_cost + fixed_cost

# Penalizing factor of 10^6 to make it so that we prioritize demand served then cost (like uber example in class)
M = 10**6
model += total_cost - M * served

**10. Add all of your constraints to your `model`.   Use python comments to document each type of constraint before you add them.**

In [30]:
# demand served at hospital cannot exceed hospital demand
for h in hospitals_dict:
    flow_in = 0
    for w in warehouses_dict:
        if(w,h) in y:
            flow_in += y[(w,h)]
    
    model += (flow_in <= hospitals_dict[h])

# total shipped out of plant can't exceed plant capcity
for p in plants_dict:
    flow_out = 0
    for w in warehouses_dict:
        if (p,w) in x:
            flow_out += x[(p,w)]
    
    model += (flow_out <= plants_dict[p])

# flow in must equal flow out of warehouses
for w in warehouses_dict:
    inflow = 0
    for p in plants_dict:
        if (p,w) in x:
            inflow += x[(p,w)]

    outflow = 0
    for h in hospitals_dict:
        if (w,h) in y:
            outflow += y[(w,h)]
    
    model += (inflow == outflow)

# warehouse can only be included if it is selected by binary decision variable
for w in warehouses_dict:
    inflow = 0
    for p in plants_dict:
        if (p,w) in x:
            inflow += x[(p,w)]

    model += (inflow <= warehouses_dict[w]['capacity'] *z[w])

<!-- BEGIN QUESTION -->

**11. Display your model with `print(model)`, check that all is OK**

In [31]:
print(model)

Vaccine_Netowrk_Max_Serve_Min_Cost:
MINIMIZE
223.0*x_plant_1_warehouse_1 + 295.0*x_plant_1_warehouse_2 + 192.0*x_plant_1_warehouse_3 + 159.0*x_plant_1_warehouse_4 + 247.0*x_plant_1_warehouse_5 + 187.0*x_plant_1_warehouse_6 + 42.0*x_plant_1_warehouse_7 + 223.0*x_plant_1_warehouse_8 + 39.0*x_plant_2_warehouse_1 + 239.0*x_plant_2_warehouse_2 + 136.0*x_plant_2_warehouse_3 + 345.0*x_plant_2_warehouse_4 + 191.0*x_plant_2_warehouse_5 + 73.0*x_plant_2_warehouse_6 + 244.0*x_plant_2_warehouse_7 + 167.0*x_plant_2_warehouse_8 + 158.0*x_plant_3_warehouse_1 + 372.0*x_plant_3_warehouse_2 + 269.0*x_plant_3_warehouse_3 + 250.0*x_plant_3_warehouse_4 + 324.0*x_plant_3_warehouse_5 + 80.0*x_plant_3_warehouse_6 + 149.0*x_plant_3_warehouse_7 + 300.0*x_plant_3_warehouse_8 + 118.0*x_plant_4_warehouse_1 + 136.0*x_plant_4_warehouse_2 + 43.0*x_plant_4_warehouse_3 + 270.0*x_plant_4_warehouse_4 + 88.0*x_plant_4_warehouse_5 + 176.0*x_plant_4_warehouse_6 + 169.0*x_plant_4_warehouse_7 + 70.0*x_plant_4_warehouse_8 + 33

<!-- END QUESTION -->

**12. Solve your optimization model and print its status and the optimal objective function value.  The optimal objective function value is 117910.0**

In [32]:
model.solve()
print('Status:', pulp.LpStatus[model.status])

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/conda/lib/python3.13/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/fff7d50bfdc946b5a43909b42918b4e2-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/fff7d50bfdc946b5a43909b42918b4e2-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 37 COLUMNS
At line 502 RHS
At line 535 BOUNDS
At line 544 ENDATA
Problem MODEL has 32 rows, 136 columns and 312 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -6.49883e+08 - 0.00 seconds
Cgl0004I processed model has 32 rows, 136 columns (8 integer (8 of which binary)) and 312 elements
Cbc0038I Initial state - 3 integers unsatisfied sum - 1.19483
Cbc0038I Pass   1: suminf.    0.00000 (0) obj. -5.49899e+08 iterations 4
Cbc0038I Solution found of -5.49899e+08
Cbc0038I Relaxing continuous gives -6.49858e+08
Cbc0038I Before mini

**13. Output the value of each of your variables at optimality.  Only print variables with non-zero value.**

In [34]:
print("Served:", pulp.value(pulp.lpSum(y[k] for k in y)))
print("Shipping cost:", pulp.value(shipping_cost))
print("Processing cost:", pulp.value(processing_cost))
print("Fixed cost:", pulp.value(fixed_cost))
print("Total cost:", pulp.value(total_cost))


Served: 650.0
Shipping cost: 99910.0
Processing cost: 13650.0
Fixed cost: 4350.0
Total cost: 117910.0


**14. Use Python to loop through each Hospital, display the total vaccine supplied to each hospital, its demand, shorfall, and percent of demand met.**

In [35]:
print("Open warehouses:")
for w, var in z.items():
    if var.value() > 0.5:
        print(" ", w)

print("\nNonzero plant->warehouse shipments:")
for (p,w), var in x.items():
    if var.value() > 1e-6:
        print(p, "->", w, ":", var.value())

print("\nNonzero warehouse->hospital shipments:")
for (w,h), var in y.items():
    if var.value() > 1e-6:
        print(w, "->", h, ":", var.value())


Open warehouses:
  warehouse_4
  warehouse_6
  warehouse_3
  warehouse_7
  warehouse_2
  warehouse_1

Nonzero plant->warehouse shipments:
plant_6 -> warehouse_4 : 100.0
plant_5 -> warehouse_2 : 100.0
plant_2 -> warehouse_1 : 60.0
plant_3 -> warehouse_6 : 150.0
plant_1 -> warehouse_6 : 40.0
plant_1 -> warehouse_3 : 40.0
plant_1 -> warehouse_7 : 50.0
plant_1 -> warehouse_1 : 70.0
plant_4 -> warehouse_3 : 40.0

Nonzero warehouse->hospital shipments:
warehouse_4 -> hospital_10 : 70.0
warehouse_4 -> hospital_7 : 30.0
warehouse_6 -> hospital_7 : 70.0
warehouse_6 -> hospital_5 : 50.0
warehouse_6 -> hospital_3 : 70.0
warehouse_3 -> hospital_2 : 40.0
warehouse_3 -> hospital_4 : 40.0
warehouse_7 -> hospital_6 : 30.0
warehouse_7 -> hospital_3 : 20.0
warehouse_2 -> hospital_8 : 90.0
warehouse_2 -> hospital_9 : 10.0
warehouse_1 -> hospital_1 : 50.0
warehouse_1 -> hospital_4 : 80.0
